# KiaOmni vs SnapKV (kvpress) vs Vanilla — Live Head-to-Head

Public, reproducible comparison of three configurations of the **same model**:

| Variant | Source |
|---|---|
| **Vanilla** | Full KV cache, no eviction (upper bound) |
| **KiaOmni-Gaussian** | [`kiaomni`](https://github.com/Aliw02/kiaomni) — training-free, budget-exact eviction |
| **SnapKV** | [`kvpress.SnapKVPress`](https://github.com/NVIDIA/kvpress) — NVIDIA's reference implementation of arXiv:2404.14469 |

**Model:** Qwen2.5-7B-Instruct, 4-bit NF4 · **Context:** ~4K tokens · **Backend:** SDPA · **Decoding:** greedy

**Tasks:** single-needle retrieval · multi-needle retrieval · reasoning (variable tracking) · summarization (key-fact coverage)

**Budgets:** 98, 128, 256, 512 retained tokens. Budget matching is **exact**: for each prompt, kvpress's
`compression_ratio` is computed per-sample so that `int(prompt_len * (1 - ratio)) == budget`, i.e. SnapKV and
KiaOmni keep *exactly* the same number of KV positions. SnapKV runs with the kvpress reference defaults
(`window_size=64`, `kernel_size=5`).

**How to run on Kaggle:** Settings → Accelerator = GPU (T4 / P100) · Internet = ON · Run All.

Set `VERBOSE = True` in the config cell to print every answer at every budget plus generation-PPL.
With `VERBOSE = False` only budgets 128 and 512 are printed (all budgets are still *measured* and appear
in the final tables and CSV).

In [ ]:
# --- Installation (Kaggle: Internet must be ON) -------------------------------
# Order matters:
#   1) kvpress + runtime deps
#   2) force transformers into the window compatible with BOTH libraries
#      (kvpress needs >=4.56,<5.3 ; kiaomni needs >=4.50,<5.0  ->  pin <5.0)
#   3) kiaomni with --no-deps so pip does not touch the preinstalled CUDA torch
%pip install -q kvpress bitsandbytes scipy
%pip install -q -U "transformers>=4.56,<5.0" "accelerate<2"
%pip install -q --no-deps git+https://github.com/Aliw02/kiaomni.git

import torch, transformers
import kvpress, kiaomni, bitsandbytes
v = transformers.__version__
assert v >= "4.56" and v < "5.0", f"transformers {v} outside the 4.56-5.0 compatibility window"
print(f"torch {torch.__version__} | transformers {v} | kiaomni {kiaomni.__version__}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (CPU run will be very slow)")

In [ ]:
# --- Configuration -------------------------------------------------------------
VERBOSE = False            # True: print every budget + generation-PPL. False: print budgets {128, 512} only.

MODEL_NAME      = "Qwen/Qwen2.5-7B-Instruct"
CONTEXT_TOKENS  = 4000     # target prompt length in tokens (Kaggle-friendly 4K)
BUDGETS         = [98, 128, 256, 512]
PRINT_BUDGETS   = {128, 512}   # used when VERBOSE is False

N_SINGLE   = 10            # single-needle questions
N_MULTI    = 10            # multi-needle questions (3 needles each)
N_REASON   = 10            # variable-tracking questions
N_SUMMARY  = 3             # summarization documents

MAX_NEW = {"single": 24, "multi": 96, "reason": 24, "summary": 220}
SEED = 42

# SnapKV reference defaults from kvpress (do NOT change: this is the published baseline)
SNAPKV_WINDOW  = 64
SNAPKV_KERNEL  = 5

In [ ]:
# --- Load model once (4-bit NF4, SDPA) ------------------------------------------
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb,
    device_map="auto",
    attn_implementation="sdpa",   # T4/P100 have no FlashAttention-2; SDPA is supported by both libraries
    torch_dtype=torch.float16,
)
model.eval()

if torch.cuda.is_available():
    print(f"Model loaded. VRAM after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# --- Engine: the three run modes, exact budget matching, metrics ----------------
import time, math, gc, re, logging
import torch
from kvpress import SnapKVPress
from kiaomni import apply_kiaomni, remove_kiaomni

# kiaomni logs an SDPA advisory on every apply_kiaomni call; with ~260 applies it would flood the log.
# SDPA is verified working (hook-based saliency observes q/k projections before the fused kernel).
logging.getLogger("kiaomni").setLevel(logging.ERROR)

def ratio_for_budget(prompt_len: int, budget: int) -> float:
    """kvpress keeps n_kept = int(prompt_len * (1 - ratio)).
    The +0.5 offset makes int() floor to exactly `budget`."""
    r = 1.0 - (budget + 0.5) / prompt_len
    assert 0.0 <= r < 1.0, f"prompt_len={prompt_len} too short for budget={budget}"
    return r

def build_inputs(context: str, question: str):
    messages = [{"role": "user", "content": f"{context}\n\n{question}"}]
    ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
    return ids, torch.ones_like(ids)

def count_tokens(text: str) -> int:
    return len(tok(text, add_special_tokens=False).input_ids)

@torch.no_grad()
def _generate(input_ids, attention_mask, max_new):
    dev = next(model.parameters()).device
    input_ids, attention_mask = input_ids.to(dev), attention_mask.to(dev)
    cuda = torch.cuda.is_available()
    if cuda:
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    out = model.generate(
        input_ids=input_ids, attention_mask=attention_mask,
        max_new_tokens=max_new, do_sample=False,
        output_scores=True, return_dict_in_generate=True,
        pad_token_id=tok.pad_token_id,
    )
    if cuda:
        torch.cuda.synchronize()
    dt = time.perf_counter() - t0

    n_new   = len(out.scores)
    new_ids = out.sequences[0, -n_new:]                    # robust even if the wrapper alters the prompt part
    logps   = [torch.log_softmax(s[0].float(), -1)[new_ids[i]].item()
               for i, s in enumerate(out.scores)]
    ppl  = math.exp(-sum(logps) / len(logps)) if logps else float("nan")
    vram = torch.cuda.max_memory_allocated() / 1e9 if cuda else 0.0
    text = tok.decode(new_ids, skip_special_tokens=True).strip()
    return {"text": text, "ppl": ppl, "tps": n_new / dt, "vram": vram, "secs": dt, "n_new": n_new}

def run_vanilla(ids, mask, max_new):
    return _generate(ids, mask, max_new)

def run_kiaomni(ids, mask, budget, max_new):
    apply_kiaomni(model, policy="kiaomni_gaussian", budget=budget)   # library defaults: n_sink=16, recency=32
    try:
        return _generate(ids, mask, max_new)
    finally:
        remove_kiaomni(model)

def run_snapkv(ids, mask, budget, max_new):
    press = SnapKVPress(
        compression_ratio=ratio_for_budget(ids.shape[1], budget),
        window_size=SNAPKV_WINDOW, kernel_size=SNAPKV_KERNEL,
    )
    with press(model):
        return _generate(ids, mask, max_new)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# --- printing helpers -----------------------------------------------------------
def fmt_metrics(res):
    s = f"{res['tps']:5.1f} tok/s | {res['vram']:.2f} GB"
    if VERBOSE:
        s += f" | GenPPL {res['ppl']:7.2f}"
    return s

def print_run(label, res, mark, clip=170):
    txt = " ".join(res["text"].split())
    if len(txt) > clip:
        txt = txt[: clip - 3] + "..."
    print(f"  {label:<24} {mark}  [{fmt_metrics(res)}]")
    print(f"  {'':<24}    -> {txt}")

def should_print(budget):
    return VERBOSE or budget in PRINT_BUDGETS

In [ ]:
# --- Task generators (deterministic, seeded) ------------------------------------
import random

_SUBJ = ["The committee", "A regional survey", "The maintenance crew", "An early prototype",
         "The northern facility", "A visiting delegation", "The archive department", "Local observers",
         "The pilot program", "A follow-up study", "The logistics team", "An internal memo",
         "The harbor authority", "A quarterly audit", "The training division", "Field engineers"]
_VERB = ["reported", "confirmed", "documented", "reviewed", "scheduled", "postponed",
         "evaluated", "inspected", "catalogued", "summarized", "approved", "recorded"]
_OBJ  = ["minor adjustments to the ventilation schedule", "a gradual rise in afternoon foot traffic",
         "the relocation of two storage containers", "routine calibration of the measurement rigs",
         "an updated rotation plan for night shifts", "the replacement of worn signage near gate three",
         "consistent humidity readings across all halls", "a backlog of paperwork from the previous quarter",
         "slower than expected delivery of spare parts", "general satisfaction with the new canteen layout",
         "uneven wear on the loading dock surface", "stable energy consumption throughout the week"]

def _sentence(rng):
    return f"{rng.choice(_SUBJ)} {rng.choice(_VERB)} {rng.choice(_OBJ)}."

def build_haystack(rng, reserved_tokens):
    """Return a list of filler sentences totalling ~CONTEXT_TOKENS - reserved_tokens."""
    target = CONTEXT_TOKENS - reserved_tokens
    sents, step = [], 40
    while count_tokens(" ".join(sents)) < target:
        sents.extend(_sentence(rng) for _ in range(step))
    while count_tokens(" ".join(sents)) > target and sents:
        sents.pop()
    return sents

def _insert(sents, items):
    """items: list of (depth in 0..1, sentence). Returns joined context."""
    out = list(sents)
    for depth, s in sorted(items, key=lambda x: x[0], reverse=True):
        out.insert(int(depth * len(out)), s)
    return " ".join(out)

DEPTHS = [0.10, 0.30, 0.50, 0.70, 0.90]

def make_single(i):
    rng  = random.Random(SEED + 100 + i)
    code = rng.randint(100000, 999999)
    depth = DEPTHS[i % len(DEPTHS)]
    needle = f"Note for the security desk: the secret access code for vault {chr(65 + i % 26)} is {code}."
    q = "What is the secret access code mentioned in the document? Answer with the number only."
    ctx = _insert(build_haystack(rng, reserved_tokens=80), [(depth, needle)])
    gold = str(code)
    def check(ans):
        ok = re.search(rf"\b{gold}\b", ans) is not None
        return ok, ("found" if ok else "missing")
    return {"ctx": ctx, "q": q, "gold": gold, "check": check,
            "info": f"code={gold} @ depth {int(depth*100)}%"}

def make_multi(i):
    rng = random.Random(SEED + 200 + i)
    code   = rng.randint(100000, 999999)
    locker = rng.randint(100, 999)
    name   = rng.choice(["FALCON", "GRANITE", "MERIDIAN", "COBALT", "JUNIPER",
                         "VERTEX", "HARBOR", "QUARTZ", "SABLE", "ORCHID"])
    needles = [
        (0.20, f"For the record, the access code is {code}."),
        (0.50, f"Equipment update: the spare keys were moved to locker {locker}."),
        (0.80, f"Personnel note: the liaison operates under the codename {name}."),
    ]
    q = ("The document hides three secret values: an access code, a locker number, "
         "and a codename. List all three values.")
    ctx  = _insert(build_haystack(rng, reserved_tokens=140), needles)
    gold = [str(code), str(locker), name]
    def check(ans):
        up = ans.upper()
        found = sum(1 for g in gold if re.search(rf"\b{g}\b", up))
        return found == 3, f"{found}/3 needles"
    return {"ctx": ctx, "q": q, "gold": ", ".join(gold), "check": check,
            "info": "3 needles @ 20/50/80%"}

def make_reason(i):
    rng  = random.Random(SEED + 300 + i)
    val  = rng.randint(10000, 99999)
    dval = rng.randint(10000, 99999)
    while dval == val:
        dval = rng.randint(10000, 99999)
    A, B, C, D = (f"V{rng.randint(10,99)}{c}" for c in "ABCD")
    X, Y       = (f"V{rng.randint(10,99)}{c}" for c in "XY")
    chain = [
        (0.15, f"Assignment log: variable {A} was set to {val}."),
        (0.40, f"Assignment log: variable {B} was set to the value of {A}."),
        (0.60, f"Assignment log: variable {C} was set to the value of {B}."),
        (0.85, f"Assignment log: variable {D} was set to the value of {C}."),
        (0.25, f"Assignment log: variable {X} was set to {dval}."),               # distractor chain
        (0.70, f"Assignment log: variable {Y} was set to the value of {X}."),
    ]
    q = (f"Track the variable assignments in the document. "
         f"What is the final value of variable {D}? Answer with the number only.")
    ctx  = _insert(build_haystack(rng, reserved_tokens=180), chain)
    gold = str(val)
    def check(ans):
        hit  = re.search(rf"\b{gold}\b", ans) is not None
        dist = re.search(rf"\b{dval}\b", ans) is not None
        ok = hit and not dist
        return ok, ("correct" if ok else ("distractor value" if dist else "missing"))
    return {"ctx": ctx, "q": q, "gold": gold, "check": check,
            "info": f"4-hop chain, answer {gold}, distractor {dval}"}

def make_summary(i):
    rng = random.Random(SEED + 400 + i)
    company = rng.choice(["Altavera Systems", "Norwind Logistics", "Helios Materials"])
    facts = {
        "revenue":  f"{rng.randint(11, 48)}.{rng.randint(1,9)} million",
        "headcount": str(rng.randint(180, 950)),
        "churn":    f"{rng.randint(2, 9)}.{rng.randint(0,9)}%",
        "nps":      str(rng.randint(31, 78)),
        "runway":   f"{rng.randint(9, 30)} months",
        "city":     rng.choice(["Riga", "Porto", "Tallinn", "Valencia"]),
        "defects":  f"{rng.randint(1, 6)}.{rng.randint(0,9)} per thousand units",
        "launch":   rng.choice(["March", "June", "September", "November"]),
    }
    sections = [
        f"QUARTERLY REVIEW - {company}.",
        f"Finance. Total revenue for the quarter reached {facts['revenue']} dollars, and the cash runway now stands at {facts['runway']}.",
        f"People. Headcount closed at {facts['headcount']} employees after the {facts['city']} office expansion.",
        f"Customers. Monthly churn was {facts['churn']} while the net promoter score reached {facts['nps']}.",
        f"Operations. The defect rate improved to {facts['defects']}, and the next product launch is scheduled for {facts['launch']}.",
    ]
    rng2  = random.Random(SEED + 500 + i)
    sents = build_haystack(rng2, reserved_tokens=count_tokens(" ".join(sections)) + 80)
    block = len(sents) // len(sections)
    woven = []
    for k, sec in enumerate(sections):
        woven.append(sec)
        woven.extend(sents[k * block:(k + 1) * block])
    ctx = " ".join(woven)
    q = ("Summarize this quarterly review in 5-8 sentences. "
         "Include every concrete figure: revenue, headcount, churn, NPS, runway, "
         "expansion city, defect rate, and launch month.")
    keys = list(facts.values())
    def check(ans):
        up = ans.upper()
        hits = sum(1 for v in keys if v.upper() in up)
        return hits / len(keys), f"{hits}/{len(keys)} key facts"
    return {"ctx": ctx, "q": q, "gold": "; ".join(f"{k}={v}" for k, v in facts.items()),
            "check": check, "info": f"{company}, 8 planted facts"}

In [ ]:
# --- Run everything --------------------------------------------------------------
results = []   # one row per generation

TASKS = [
    ("single",  make_single,  N_SINGLE),
    ("multi",   make_multi,   N_MULTI),
    ("reason",  make_reason,  N_REASON),
    ("summary", make_summary, N_SUMMARY),
]
total_runs = sum(n for _, _, n in TASKS) * (1 + 2 * len(BUDGETS))
print(f"Total generations: {total_runs} "
      f"({sum(n for *_, n in TASKS)} samples x (1 vanilla + 2 methods x {len(BUDGETS)} budgets))\n")

def record(task, sid, method, budget, res, score, detail):
    results.append({"task": task, "sample": sid, "method": method,
                    "budget": budget if budget is not None else 0,
                    "score": float(score), "detail": detail,
                    "tps": res["tps"], "vram": res["vram"], "ppl": res["ppl"],
                    "text": res["text"]})

t_start = time.perf_counter()
done = 0

for task, maker, n in TASKS:
    print("=" * 100)
    print(f"TASK: {task.upper()}  ({n} samples)")
    print("=" * 100)
    for sid in range(n):
        s = maker(sid)
        ids, mask = build_inputs(s["ctx"], s["q"])
        plen = ids.shape[1]
        print(f"\n[{task} #{sid + 1}]  prompt = {plen} tokens  |  {s['info']}")
        print(f"  GOLD: {s['gold']}")
        print("-" * 100)

        is_summary = task == "summary"

        # ---- vanilla (once per sample) ----
        res = run_vanilla(ids, mask, MAX_NEW[task]); done += 1
        sc, det = s["check"](res["text"])
        mark = f"{sc * 100:5.1f}% facts" if is_summary else ("PASS" if sc else "FAIL")
        if is_summary:
            print(f"  Vanilla (full cache)      {mark} ({det})  [{fmt_metrics(res)}]")
            print("    " + res["text"].replace("\n", "\n    "))
        else:
            print_run("Vanilla (full cache)", res, f"{mark} ({det})")
        record(task, sid, "Vanilla", None, res, sc, det)

        # ---- both methods at every budget ----
        for budget in BUDGETS:
            for mname, runner in (("KiaOmni-Gaussian", run_kiaomni), ("SnapKV", run_snapkv)):
                res = runner(ids, mask, budget, MAX_NEW[task]); done += 1
                sc, det = s["check"](res["text"])
                mark = f"{sc * 100:5.1f}% facts" if is_summary else ("PASS" if sc else "FAIL")
                if should_print(budget):
                    label = f"{mname}  B={budget}"
                    if is_summary:
                        print(f"  {label:<24} {mark} ({det})  [{fmt_metrics(res)}]")
                        print("    " + res["text"].replace("\n", "\n    "))
                    else:
                        print_run(label, res, f"{mark} ({det})")
                record(task, sid, mname, budget, res, sc, det)
        cleanup()

        if done and sid == 0 and task == "single":
            per = (time.perf_counter() - t_start) / done
            print(f"\n  [timing] {per:.1f}s per generation -> "
                  f"estimated total ~{per * total_runs / 60:.0f} min")

print(f"\nDone. {done} generations in {(time.perf_counter() - t_start) / 60:.1f} min.")

In [ ]:
# --- Final tables, chart, CSV -----------------------------------------------------
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(results)
df["score_pct"] = df["score"] * 100
df["config"] = df.apply(
    lambda r: "Vanilla (full)" if r["method"] == "Vanilla" else f"{r['method']} B={int(r['budget'])}", axis=1)

row_order = (["Vanilla (full)"]
             + [f"{m} B={b}" for b in BUDGETS for m in ("KiaOmni-Gaussian", "SnapKV")])

print("ACCURACY / COVERAGE (%) -- single, multi, reason = accuracy; summary = key-fact coverage")
acc = (df.pivot_table(index="config", columns="task", values="score_pct", aggfunc="mean")
         .reindex(row_order)[["single", "multi", "reason", "summary"]].round(1))
acc["needle_mean"] = acc[["single", "multi", "reason"]].mean(axis=1).round(1)
print(acc.to_string(), "\n")

print("EFFICIENCY (means over all generations)")
eff_cols = ["tps", "vram"] + (["ppl"] if VERBOSE else [])
eff = df.groupby("config")[eff_cols].mean().reindex(row_order).round(2)
eff.columns = ["tokens/sec", "peak VRAM (GB)"] + (["Gen-PPL"] if VERBOSE else [])
print(eff.to_string(), "\n")

# --- chart: mean retrieval+reasoning accuracy per budget --------------------------
fig, ax = plt.subplots(figsize=(9, 4.5))
needle = df[df["task"].isin(["single", "multi", "reason"])]
van = needle[needle["method"] == "Vanilla"]["score_pct"].mean()
width = 0.36
xs = range(len(BUDGETS))
for off, (m, color) in enumerate([("KiaOmni-Gaussian", "#3fa7d6"), ("SnapKV", "#e05c5c")]):
    ys = [needle[(needle["method"] == m) & (needle["budget"] == b)]["score_pct"].mean() for b in BUDGETS]
    ax.bar([x + (off - 0.5) * width for x in xs], ys, width, label=m, color=color)
    for x, y in zip(xs, ys):
        ax.text(x + (off - 0.5) * width, y + 1, f"{y:.0f}", ha="center", fontsize=9)
ax.axhline(van, ls="--", c="gray", label=f"Vanilla ({van:.0f}%)")
ax.set_xticks(list(xs)); ax.set_xticklabels([f"B={b}" for b in BUDGETS])
ax.set_ylabel("Mean accuracy % (single + multi + reason)")
ax.set_title(f"{MODEL_NAME} (NF4, ~{CONTEXT_TOKENS} tok context) -- exact-budget comparison")
ax.set_ylim(0, 105); ax.legend()
plt.tight_layout(); plt.show()

df.drop(columns=["score_pct"]).to_csv("kiaomni_vs_snapkv_results.csv", index=False)
print("Full per-generation log (including all answer texts) saved to kiaomni_vs_snapkv_results.csv")

## Notes and caveats

- **Small N by design.** This is a live, fully-printed demo (10/10/10 questions + 3 documents), not a
  paper-scale evaluation — confidence intervals are wide. The paper-scale numbers (61,681 LLM-judged
  samples) live in the [KiaOmni repo](https://github.com/Aliw02/kiaomni).
- **SnapKV is the NVIDIA `kvpress` reference implementation**, run at its published defaults
  (`window_size=64`, `kernel_size=5`), GQA-correct, with question-aware compression (the question is part
  of the prompt at prefill, which is SnapKV's intended strongest setting).
- **Budget matching is exact per sample**: both methods retain the same number of KV positions.
  Vanilla keeps the full cache and is the soft upper bound.
- **Generation-PPL** (VERBOSE mode) is the perplexity of the model over its *own* greedy tokens while
  answering from a compressed cache. It measures fluency/confidence under eviction and is comparable
  across the three variants — it is **not** WikiText perplexity.
- Greedy decoding, fp16 compute, NF4 weights, SDPA backend. Tasks are synthetic and seeded —
  rerunning reproduces the exact same prompts.